# Lab 3 – Hybrid search + Rerank

Chunk `structure` của Lab 2 (breadcrumb + nội dung). So sánh trên cùng golden set:

| Phương pháp | Mô tả |
|---|---|
| `dense` | BGE-M3 dense, pgvector HNSW (m=16, ef_construction=64, ef_search=40), cosine |
| `bm25` | BM25 trên `embed_text`, tách từ tiếng Việt bằng pyvi (`bảo_hiểm` là 1 từ) |
| `bm25_nd` | BM25 sau khi **bỏ dấu** cả văn bản lẫn câu hỏi, tách theo âm tiết |
| `hybrid` | RRF(dense, bm25), k=60 |
| `hybrid3` | RRF(dense, bm25, bm25_nd) |
| `+rerank` | lấy 50 ứng viên từ RRF, rerank bằng cross-encoder `bge-reranker-v2-m3` |
| `+filter` | nhận diện sản phẩm trong câu hỏi (alias trong manifest), chỉ tìm trong `doc_id` đó |
| `+rewrite` | LLM thêm dấu, mở rộng viết tắt (NĐBH, STBH…) trước khi tìm |

Chạy trên **2 bộ câu hỏi**: câu gốc (20) và cùng 20 câu **đã bỏ dấu** (mô phỏng người dùng gõ không dấu).

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)

from ragbv.config import load_manifest, REPORTS
from ragbv import chunking as C, embedding as EMB, metrics as M, store, retrieval as R
from ragbv.bm25 import strip_diacritics
from ragbv.golden import load_golden, Item

docs = load_manifest()
chunks = C.load_chunks("structure")
golden = [g for g in load_golden() if g.in_scope]
golden_nd = [Item(**{**g.__dict__, "question": strip_diacritics(g.question)}) for g in golden]
print(len(chunks), "chunk |", len(golden), "câu |", golden_nd[0].question)

398 chunk | 20 câu | Phi bao hiem dieu khoan chinh (dieu tri noi tru) goi Bac cua Bao Viet An Gia cho nguoi 35 tuoi la bao nhieu mot nam?


## 1. Nạp vào pgvector

In [2]:
conn = store.connect()
store.init(conn)
vecs = EMB.encode([c.embed_text for c in chunks])
store.upsert(conn, chunks, vecs)
print(conn.execute("SELECT strategy, count(*) FROM chunks GROUP BY strategy").fetchall())
print(conn.execute("SELECT indexdef FROM pg_indexes WHERE indexname = 'chunks_hnsw'").fetchone()[0])
print("pgvector", conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0])

[('structure', 397)]
CREATE INDEX chunks_hnsw ON public.chunks USING hnsw (embedding vector_cosine_ops) WITH (m='16', ef_construction='64')
pgvector 0.8.7


### Overfiltering: lọc metadata làm HNSW trả về thiếu kết quả
pgvector áp `WHERE` **sau** khi duyệt `ef_search` ứng viên gần nhất. Lọc theo 1 tài liệu (khoảng 14% số dòng) thì chỉ còn vài dòng, dù `LIMIT 20`. Từ pgvector 0.8, `hnsw.iterative_scan` tiếp tục duyệt đến khi đủ.

Bảng chỉ có 398 dòng nên **mặc định planner không dùng HNSW** (quét + sắp xếp chính xác, luôn đúng). Để thấy hành vi ở quy mô lớn, buộc dùng index (`force_index`).

In [3]:
qv = EMB.encode(["Thời gian chờ quyền lợi thai sản"])[0]
avsk = ["bvl-avsk-2022"]
with conn.transaction():
    plan = conn.execute("EXPLAIN SELECT chunk_id FROM chunks WHERE strategy='structure' AND doc_id = ANY(%s) "
                        "ORDER BY embedding <=> %s LIMIT 20", [avsk, qv]).fetchall()
print("Kế hoạch mặc định:", " -> ".join(p[0].split("(")[0].strip() for p in plan[:3]))
rows = [{"plan": "mặc định (planner tự chọn)", "ef_search": 40, "iterative_scan": "off",
         "rows_returned (LIMIT 20)": len(store.search(conn, qv, "structure", 20, avsk))}]
for ef in (10, 40):
    for it in ("off", "relaxed_order"):
        n = len(store.search(conn, qv, "structure", 20, avsk, ef_search=ef, iterative=it, force_index=True))
        rows.append({"plan": "buộc dùng HNSW", "ef_search": ef, "iterative_scan": it, "rows_returned (LIMIT 20)": n})
n_avsk = sum(c.doc_id == "bvl-avsk-2022" for c in chunks)
print(f"AVSK có {n_avsk}/{len(chunks)} chunk")
pd.DataFrame(rows)

Kế hoạch mặc định: Limit -> ->  Sort -> Sort Key:
AVSK có 57/398 chunk


,plan,ef_search,iterative_scan,rows_returned (LIMIT 20)
0,mặc định (planner tự chọn),40,off,20
1,buộc dùng HNSW,10,off,4
2,buộc dùng HNSW,10,relaxed_order,20
3,buộc dùng HNSW,40,off,8
4,buộc dùng HNSW,40,relaxed_order,20


## 2. Đánh giá

In [4]:
ret = R.Retriever(chunks, docs, conn, "structure")

def evaluate(items, method, **kw):
    out = []
    for g in items:
        ranked = ret.run(g.question, method, k=10, **kw)
        out.append({"method": label(method, **kw), **M.score_item(g, ranked), "_ranked": ranked})
    return out

def label(method, use_filter=False, use_rewrite=False):
    return ("rewrite+" if use_rewrite else "") + method + ("+filter" if use_filter else "")

CONFIGS = [("dense", {}), ("bm25", {}), ("bm25_nd", {}), ("hybrid", {}), ("hybrid3", {}),
           ("hybrid3+rerank", {}), ("dense", {"use_filter": True}), ("hybrid3+rerank", {"use_filter": True}),
           ("hybrid3+rerank", {"use_filter": True, "use_rewrite": True})]
runs = {}
for qset, items in (("gốc", golden), ("bỏ dấu", golden_nd)):
    for m, kw in CONFIGS:
        runs[(qset, label(m, **kw))] = evaluate(items, m, **kw)
results = pd.DataFrame([{**r, "qset": q} for (q, _), rs in runs.items() for r in rs]).drop(columns=["_ranked"])
cols = ["hit@1", "hit@3", "hit@5", "hit@10", "recall@5", "recall@10", "mrr"]
order = [label(m, **kw) for m, kw in CONFIGS]
summary = results.groupby(["qset", "method"])[cols].mean().round(3)
summary.loc["gốc"].loc[order]

C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   4%|▎         | 14/393 [00:00<00:02, 130.43it/s]

Loading weights:   7%|▋         | 28/393 [00:00<00:03, 108.08it/s]

Loading weights:  10%|█         | 40/393 [00:00<00:05, 64.81it/s] 

Loading weights:  12%|█▏        | 48/393 [00:00<00:06, 50.34it/s]

... (2 outputs left out)

Loading weights:  18%|█▊        | 69/393 [00:01<00:07, 43.59it/s]

Loading weights:  19%|█▉        | 75/393 [00:01<00:07, 45.30it/s]

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 279.76it/s]

,hit@1,hit@3,hit@5,hit@10,recall@5,recall@10,mrr
method,,,,,,,
dense,0.55,0.70,0.80,0.90,0.767,0.858,0.661
bm25,0.50,0.65,0.80,0.90,0.750,0.817,0.616
bm25_nd,0.35,0.65,0.75,0.85,0.700,0.767,0.536
hybrid,0.60,0.80,0.85,0.90,0.792,0.842,0.711
hybrid3,0.55,0.85,0.90,0.90,0.842,0.842,0.696
hybrid3+rerank,0.65,0.95,0.95,0.95,0.900,0.925,0.792
dense+filter,0.55,0.75,0.90,0.90,0.867,0.883,0.679
hybrid3+rerank+filter,0.70,0.95,0.95,0.95,0.925,0.925,0.817
rewrite+hybrid3+rerank+filter,0.70,0.95,0.95,0.95,0.925,0.925,0.825


In [5]:
summary.loc["bỏ dấu"].loc[order]

,hit@1,hit@3,hit@5,hit@10,recall@5,recall@10,mrr
method,,,,,,,
dense,0.10,0.15,0.30,0.50,0.300,0.500,0.178
bm25,0.00,0.00,0.05,0.10,0.025,0.075,0.020
bm25_nd,0.35,0.65,0.75,0.85,0.700,0.767,0.536
hybrid,0.05,0.10,0.15,0.30,0.125,0.300,0.106
hybrid3,0.10,0.25,0.45,0.70,0.450,0.700,0.238
hybrid3+rerank,0.20,0.60,0.80,0.85,0.750,0.800,0.436
dense+filter,0.10,0.25,0.45,0.60,0.450,0.600,0.224
hybrid3+rerank+filter,0.20,0.50,0.80,0.80,0.750,0.750,0.414
rewrite+hybrid3+rerank+filter,0.70,0.95,0.95,0.95,0.925,0.925,0.817


### Theo loại câu hỏi (recall@5, câu gốc)

In [6]:
results[results.qset == "gốc"].pivot_table(index="type", columns="method", values="recall@5", aggfunc="mean").round(2)[order]

method,dense,bm25,bm25_nd,hybrid,hybrid3,hybrid3+rerank,dense+filter,hybrid3+rerank+filter,rewrite+hybrid3+rerank+filter
type,,,,,,,,,
comparison,0.17,0.25,0.25,0.17,0.42,0.75,0.67,1.00,1.00
exclusion,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
fact,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
multi_condition,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
no_diacritics,0.00,0.25,0.75,0.25,0.50,0.75,0.50,0.75,0.75
table_lookup,0.80,0.60,0.20,0.80,0.80,0.80,0.80,0.80,0.80
versioning,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


### Từng câu: hạng bằng chứng (câu gốc)

In [7]:
pick = ["dense", "bm25", "bm25_nd", "hybrid3", "hybrid3+rerank", "hybrid3+rerank+filter", "rewrite+hybrid3+rerank+filter"]
results[results.qset == "gốc"].pivot(index="id", columns="method", values="ranks")[pick]

method,dense,bm25,bm25_nd,hybrid3,hybrid3+rerank,hybrid3+rerank+filter,rewrite+hybrid3+rerank+filter
id,,,,,,,
G01,[2],[3],[None],[2],[3],[3],[2]
G02,[1],[1],[6],[1],[1],[1],[1]
G03,[None],[None],[None],[None],[None],[None],[None]
G04,[1],[1],[2],[1],[1],[1],[1]
G05,[1],[1],[2],[1],[1],[1],[1]
G06,"[1, 1]","[1, 1]","[2, 2]","[1, 1]","[1, 1]","[1, 1]","[1, 1]"
G07,[1],[4],[1],[1],[2],[2],[2]
G08,[1],[1],[1],[1],[1],[1],[1]
G09,"[1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]","[1, 1]"


In [8]:
results[results.qset == "bỏ dấu"].pivot(index="id", columns="method", values="ranks")[pick]

method,dense,bm25,bm25_nd,hybrid3,hybrid3+rerank,hybrid3+rerank+filter,rewrite+hybrid3+rerank+filter
id,,,,,,,
G01,[8],[None],[None],[None],[2],[2],[3]
G02,[1],[None],[6],[4],[1],[1],[1]
G03,[None],[None],[None],[None],[None],[None],[None]
G04,[None],[None],[2],[8],[2],[2],[1]
G05,[1],[7],[2],[2],[3],[3],[1]
G06,"[10, 10]","[None, None]","[2, 2]","[3, 3]","[2, 2]","[2, 2]","[1, 1]"
G07,[None],[None],[1],[5],[2],[2],[2]
G08,[4],[None],[1],[5],[3],[4],[1]
G09,"[7, 7]","[None, None]","[1, 1]","[4, 4]","[None, None]","[None, None]","[1, 1]"


### Viết lại câu hỏi bằng LLM

In [9]:
for g in [x for x in golden if x.id in ("G19", "G20")] + [x for x in golden_nd if x.id in ("G01", "G11")]:
    print(f"{g.id}: {g.question}\n   -> {R.rewrite(g.question)}")

G19: thoi han bao hiem cua tu ky tai tuc hang thang dai ichi la bao lau, NDBH toi da bao nhieu tuoi
   -> Thời hạn bảo hiểm của Tự Ký Tái Tục Hàng Tháng Dai Ichi là bao lâu, Người được bảo hiểm tối đa bao nhiêu tuổi?
G20: STBH bao hiem tai nan ca nhan trong An Gia toi da bao nhieu va phi tinh the nao
   -> Số tiền bảo hiểm bảo hiểm tai nạn cá nhân trong An Gia tối đa bao nhiêu và phí tính thế nào?
G01: Phi bao hiem dieu khoan chinh (dieu tri noi tru) goi Bac cua Bao Viet An Gia cho nguoi 35 tuoi la bao nhieu mot nam?
   -> Phí bảo hiểm điều khoản chính (điều trị nội trú) gói Bac của Bảo Việt An Gia cho người 35 tuổi là bao nhiêu một năm?
G11: Thoi gian cho cua quyen loi thai san trong san pham An Vui Song Khoe la bao lau?
   -> Thời gian chờ của quyền lợi thai sản trong sản phẩm An Vui Sống Khỏe là bao lâu?


## 3. Độ trễ từng bước (ms, 20 câu gốc, GPU RTX 3050 Ti 4GB)
Embedding câu hỏi đo trực tiếp bằng model (bỏ qua cache). Rerank tính trên 50 ứng viên.

In [10]:
lat = {"embed_query": [], "pgvector": [], "bm25": [], "bm25_nd": [], "rrf": []}
model = EMB.model()
R.rerank("khởi động", chunks[:4])  # lần gọi đầu nạp model lên GPU, không tính vào độ trễ
for g in golden:
    t0 = time.perf_counter(); qv = model.encode([g.question], max_length=1024)["dense_vecs"][0]; lat["embed_query"].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); d = store.search(conn, qv, "structure", 50); lat["pgvector"].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); b = ret.bm25["bm25"].search(g.question, 50); lat["bm25"].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); bn = ret.bm25["bm25_nd"].search(g.question, 50); lat["bm25_nd"].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); fused = R.rrf([[x for x, _ in d], [x for x, _ in b], [x for x, _ in bn]]); lat["rrf"].append(time.perf_counter() - t0)
    for n in (10, 20, 50):
        cands = [ret.chunks[cid] for cid, _ in fused[:n]]
        t0 = time.perf_counter(); R.rerank(g.question, cands); lat.setdefault(f"rerank_{n}", []).append(time.perf_counter() - t0)
latency = pd.DataFrame({k: {"p50": np.percentile(v, 50) * 1000, "p95": np.percentile(v, 95) * 1000} for k, v in lat.items()}).T.round(1)
latency

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 2076.59it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   1%|▏         | 5/391 [00:02<02:54,  2.21it/s]

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 170.36it/s]

,p50,p95
embed_query,30.3,126.6
pgvector,7.1,9.4
bm25,2.2,2.9
bm25_nd,2.1,3.1
rrf,0.1,0.1
rerank_10,471.4,499.5
rerank_20,920.4,996.4
rerank_50,2371.9,2501.4


### Đánh đổi: số ứng viên đưa vào rerank
Rerank là bước chậm nhất. Ít ứng viên hơn thì nhanh hơn, nhưng có thể bỏ sót chunk đúng nằm sâu trong danh sách RRF.

In [11]:
trade = []
for n in (10, 20, 50):
    rs = [M.score_item(g, ret.run(g.question, "hybrid3+rerank", k=10, n_cand=n, use_filter=True)) for g in golden]
    trade.append({"n_cand": n, "recall@5": np.mean([r["recall@5"] for r in rs]), "mrr": np.mean([r["mrr"] for r in rs]),
                  "rerank_p50_ms": latency.loc[f"rerank_{n}", "p50"]})
trade = pd.DataFrame(trade).round(3)
trade

,n_cand,recall@5,mrr,rerank_p50_ms
0,10,0.892,0.825,471.4
1,20,0.900,0.767,920.4
2,50,0.925,0.817,2371.9


## 4. Báo cáo chi tiết từng câu và lưu kết quả

In [12]:
show = ["dense", "hybrid3", "hybrid3+rerank", "hybrid3+rerank+filter", "rewrite+hybrid3+rerank+filter"]
for qset, items, name in (("gốc", golden, "lab3_retrieval_details.md"), ("bỏ dấu", golden_nd, "lab3_retrieval_details_nodiac.md")):
    ranked = {m: [r["_ranked"] for r in runs[(qset, m)]] for m in show}
    (REPORTS / name).write_text(M.details_markdown(items, ranked, f"Lab 3 – Kết quả truy xuất từng câu ({qset})"), encoding="utf-8")

out = {"summary": summary.reset_index().to_dict("records"),
       "by_type": results[results.qset == "gốc"].pivot_table(index="type", columns="method", values="recall@5", aggfunc="mean").round(3).reset_index().to_dict("records"),
       "per_question": results.assign(ranks=results["ranks"].astype(str)).to_dict("records"),
       "overfilter": rows, "latency_ms": latency.reset_index().to_dict("records"),
       "rerank_tradeoff": trade.to_dict("records"),
       "rewrites": {**{g.id: {"q": g.question, "rewritten": R.rewrite(g.question)} for g in golden},
                    **{g.id + "_nd": {"q": g.question, "rewritten": R.rewrite(g.question)} for g in golden_nd}}}
(REPORTS / "lab3_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
